# Дообучение эмбеддера для RAG-системы по российскому законодательству (ПДД, штрафы, юр. домен)

**Цель ноутбука:** дообучить SOTA-эмбеддер на собственном русскоязычном legal-домене с помощью библиотеки `sentence-transformers`. Архитектура — bi-encoder, обучаемый контрастивным лоссом с in-batch и hard negatives (`MultipleNegativesRankingLoss` / `CachedMultipleNegativesRankingLoss`).

**Среда:** Google Colab. Рекомендуется GPU `T4` (16GB) или `A100` (40GB). Для `T4` оптимальна базовая модель `intfloat/multilingual-e5-base` или `Qwen/Qwen3-Embedding-0.6B`; для `A100` можно брать `BAAI/bge-m3` (568M) или `Qwen/Qwen3-Embedding-4B`.

**Что делает ноутбук:**
1. Ставит зависимости и поднимает Colab-сессию.
2. Грузит датасет в формате `(query, positive, negative)` или `(query, positive)` — оба варианта поддержаны.
3. Делит на train/eval, готовит `InformationRetrievalEvaluator` для честных метрик retrieval (Recall@k, MRR@10, NDCG@10).
4. Бейзлайн: считает метрики **до** дообучения.
5. Дообучает модель (с прогрев-шагами, AdamW, FP16/BF16, контрастивный лосс).
6. Считает метрики **после** дообучения и сравнивает.
7. Сохраняет модель локально и в HuggingFace Hub (опционально).

**Где это вписывается в проект:** этап `retriever` в `RAG_for_law` — заменяет дефолтный энкодер на дообученный, после чего пайплайн ранжирования (Recall@k / NDCG@k) должен показать прирост на твоём домене.

## 0. Окружение Colab

Проверяем GPU и ставим зависимости. На свежем Colab `sentence-transformers >= 3.0` уже поддерживает новый `SentenceTransformerTrainer` (Trainer-API в стиле HF). Это и используем.

In [ ]:
!nvidia-smi

In [ ]:
# Ставим свежие версии. sentence-transformers >= 3.3 + transformers >= 4.51 — 
# минимум для поддержки Qwen3-Embedding и нового Trainer-API.
!pip install -q -U "sentence-transformers>=3.3.0" "transformers>=4.51.0" "datasets>=2.19.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

In [ ]:
import os
import json
import random
import logging
from pathlib import Path

import torch
import numpy as np
from datasets import Dataset, load_dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
)
from sentence_transformers.training_args import BatchSamplers

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM, GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Выбор базовой модели

Подсказки по выбору:

| Модель | Размер | Когда брать |
| --- | --- | --- |
| `intfloat/multilingual-e5-base` | 278M | Бейзлайн, влезает на T4 с batch_size=32–64. Стабильная, хорошо файн-тюнится. |
| `intfloat/multilingual-e5-large` | 560M | Если T4 хватает (batch_size=8–16 + gradient checkpointing). |
| `BAAI/bge-m3` | 568M | Сильный мультиязычный SOTA, длинный контекст (8192). На T4 — только с gradient_checkpointing и BF16/FP16. |
| `Qwen/Qwen3-Embedding-0.6B` | 0.6B | Топ multilingual MTEB на момент выпуска; instruction-aware. Требует префикса `Instruct:` для запросов. |
| `deepvk/USER-bge-m3` / `deepvk/USER-base` | ru-focused | Хорошие русско-ориентированные варианты, можно сравнить как бейзлайн. |
| `ai-forever/ru-en-RoSBERTa` | ~400M | Сильный русско-английский эмбеддер из ruMTEB. |

Для первого прохода рекомендую `intfloat/multilingual-e5-base` — стабильный SOTA-бейзлайн, который точно влезет в T4 и быстро обучится. Когда пайплайн заработает, можно подменить на BGE-M3 или Qwen3.

In [ ]:
# =========  ВЫБОР МОДЕЛИ  =========
BASE_MODEL = "intfloat/multilingual-e5-base"
# BASE_MODEL = "BAAI/bge-m3"
# BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"
# BASE_MODEL = "ai-forever/ru-en-RoSBERTa"

OUTPUT_DIR = f"./finetuned-{BASE_MODEL.split('/')[-1]}-legal-ru"

# Для E5 запросы и документы должны идти с префиксами:
#   запросы:   "query: ..."
#   документы: "passage: ..."
# Для BGE-M3 префиксы НЕ нужны.
# Для Qwen3-Embedding запросы оборачиваются в инструкцию:
#   "Instruct: Given a legal question, retrieve relevant regulation passages\nQuery: ..."

def get_prefix_fn(model_name: str):
    name = model_name.lower()
    if "e5" in name:
        return (lambda q: f"query: {q}", lambda d: f"passage: {d}")
    if "bge" in name and "m3" in name:
        return (lambda q: q, lambda d: d)
    if "qwen3-embedding" in name:
        instr = "Given a legal question in Russian, retrieve relevant passages from traffic regulations and laws"
        return (lambda q: f"Instruct: {instr}\nQuery: {q}", lambda d: d)
    if "rosberta" in name:
        # ru-en-RoSBERTa использует префиксы search_query / search_document
        return (lambda q: f"search_query: {q}", lambda d: f"search_document: {d}")
    return (lambda q: q, lambda d: d)

format_query, format_doc = get_prefix_fn(BASE_MODEL)
print("Префиксы:")
print("  query   ->", format_query("Какой штраф за превышение скорости на 25 км/ч?"))
print("  doc     ->", format_doc("Превышение установленной скорости движения на величину более 20...")[:120])

## 2. Подготовка датасета

Файн-тюн ожидает один из форматов:

**(А) Triplets** — `(anchor, positive, negative)`. Самый сильный сигнал, особенно с майнингом hard negatives.

```json
{"query": "...", "positive": "...", "negative": "..."}
```

**(Б) Pairs** — `(anchor, positive)`. С `MultipleNegativesRankingLoss` остальные позитивы в батче автоматически становятся in-batch негативами. Минимум усилий на разметку, но качество ниже, чем при наличии hard negatives.

```json
{"query": "...", "positive": "..."}
```

Ниже путь к JSONL-файлу. Если файла нет — создаётся синтетический пример, чтобы ноутбук прогонялся end-to-end (заменишь реальным датасетом).

In [ ]:
DATA_PATH = "./data/train.jsonl"   # путь к твоему датасету (JSONL)
USE_TRIPLETS = True                # True если есть колонка `negative`

# --------- Если своего датасета пока нет — сгенерируем игрушечный для прогонки ---------
if not Path(DATA_PATH).exists():
    Path("data").mkdir(exist_ok=True)
    toy = [
        {
            "query": "Какой штраф за превышение скорости на 25 км/ч?",
            "positive": "Превышение установленной скорости движения транспортного средства на величину более 20, но не более 40 км/ч влечёт наложение административного штрафа в размере пятисот рублей. — Статья 12.9 КоАП РФ, часть 2.",
            "negative": "Управление транспортным средством, не зарегистрированным в установленном порядке, влечёт наложение административного штрафа в размере от пятисот до восьмисот рублей. — Статья 12.1 КоАП РФ.",
        },
        {
            "query": "Можно ли обгонять на перекрёстке?",
            "positive": "Обгон запрещён на регулируемых перекрёстках, а также на нерегулируемых перекрёстках при движении по дороге, не являющейся главной. — пункт 11.4 ПДД РФ.",
            "negative": "При движении по дороге с двусторонним движением, имеющей четыре или более полосы, запрещается выезжать для обгона или объезда на полосу, предназначенную для встречного движения. — пункт 9.2 ПДД РФ.",
        },
        {
            "query": "Какое наказание за вождение в нетрезвом виде первый раз?",
            "positive": "Управление транспортным средством водителем, находящимся в состоянии опьянения, если такие действия не содержат уголовно наказуемого деяния, влечёт наложение административного штрафа в размере тридцати тысяч рублей с лишением права управления транспортными средствами на срок от полутора до двух лет. — Статья 12.8 КоАП РФ, часть 1.",
            "negative": "Невыполнение водителем требований об остановке транспортного средства влечёт наложение административного штрафа в размере от восьмисот до тысячи рублей. — Статья 12.25 КоАП РФ.",
        },
        {
            "query": "Где запрещена остановка?",
            "positive": "Остановка запрещается: на трамвайных путях, а также в непосредственной близости от них, если это создаст помехи движению трамваев; на железнодорожных переездах, в тоннелях, а также на эстакадах, мостах, путепроводах. — пункт 12.4 ПДД РФ.",
            "negative": "Стоянка запрещается вне населённых пунктов на проезжей части дорог, обозначенных знаком 2.1. — пункт 12.5 ПДД РФ.",
        },
    ]
    with open(DATA_PATH, "w", encoding="utf-8") as f:
        for row in toy:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"⚠️  Игрушечный датасет создан в {DATA_PATH}. Замени на свой.")

# --------- Загрузка ---------
raw = load_dataset("json", data_files=DATA_PATH, split="train")
print("Кол-во примеров:", len(raw))
print("Колонки:", raw.column_names)
print("Пример:", raw[0])

In [ ]:
# Применяем префиксы к данным и собираем итоговый Dataset.
def apply_prefixes(example):
    out = {
        "anchor": format_query(example["query"]),
        "positive": format_doc(example["positive"]),
    }
    if USE_TRIPLETS and "negative" in example and example["negative"]:
        out["negative"] = format_doc(example["negative"])
    return out

dataset = raw.map(apply_prefixes, remove_columns=raw.column_names)

# Train / eval split
ds_split = dataset.train_test_split(test_size=0.1, seed=SEED)
train_ds = ds_split["train"]
eval_ds  = ds_split["test"]
print("train:", len(train_ds), "  eval:", len(eval_ds))
print("первый train-пример:", train_ds[0])

## 3. Информационный ретривер-евалюатор

`InformationRetrievalEvaluator` гоняет настоящий retrieval: для каждого запроса ранжирует весь корпус документов и считает Recall@k, MRR@k, NDCG@k. Это **те же метрики**, что упомянуты в плане проекта (`Recall@k, Precision@k, nDCG@k, hit@k, MAP@k`).

Корпус для ретривера — все позитивы из eval-сплита. Релевантности — `{query_id: {doc_id}}`.

In [ ]:
def build_ir_evaluator(eval_dataset, name="legal-eval"):
    queries = {}    # qid -> query text (С ПРЕФИКСОМ)
    corpus  = {}    # cid -> passage text (С ПРЕФИКСОМ)
    relevant_docs = {}  # qid -> set(cid)

    for i, row in enumerate(eval_dataset):
        qid = f"q{i}"
        cid_pos = f"d{i}_pos"
        queries[qid] = row["anchor"]
        corpus[cid_pos] = row["positive"]
        relevant_docs[qid] = {cid_pos}
        if "negative" in row and row["negative"]:
            cid_neg = f"d{i}_neg"
            corpus[cid_neg] = row["negative"]
            # negative — distractor, в relevant НЕ добавляется

    return evaluation.InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        accuracy_at_k=[1, 3, 5, 10],
        precision_recall_at_k=[1, 3, 5, 10],
        mrr_at_k=[10],
        ndcg_at_k=[10],
        map_at_k=[10],
        show_progress_bar=True,
    )

ir_evaluator = build_ir_evaluator(eval_ds)

## 4. Бейзлайн: метрики ДО дообучения

Загружаем модель как есть и замеряем retrieval-метрики. Это число надо превзойти после файн-тюнинга — без этого замера нет доказательства, что дообучение помогло.

In [ ]:
model = SentenceTransformer(BASE_MODEL, device=DEVICE)
print("Макс. длина последовательности:", model.max_seq_length)
print("Размерность эмбеддинга:", model.get_sentence_embedding_dimension())

# Чтобы влезать в VRAM на длинных юридических чанках:
model.max_seq_length = min(model.max_seq_length, 512)

baseline_metrics = ir_evaluator(model)
print("\n=== БЕЙЗЛАЙН (до файн-тюнинга) ===")
for k, v in baseline_metrics.items():
    print(f"{k:55s} {v:.4f}")

## 5. Лосс

`MultipleNegativesRankingLoss` (MNRL) — стандарт для retrieval-файн-тюнинга. Идея простая: в батче из N пар `(q_i, p_i)` для каждого `q_i` все остальные `p_j (j ≠ i)` считаются негативами. С триплетами хард-негатив добавляется как дополнительный негатив к каждому запросу.

Если VRAM упирается — используем `CachedMultipleNegativesRankingLoss`: разбивает большой логический батч на mini-batches с накоплением, эффективно увеличивая размер контрастивного батча без OOM.

Размер батча для MNRL критичен: чем больше — тем сильнее сигнал (больше негативов). На T4 с E5-base реально 32–64; для BGE-M3 — 8–16 с gradient checkpointing.

In [ ]:
USE_CACHED = False  # включи, если упираешься в OOM при большом batch_size

if USE_CACHED:
    loss = losses.CachedMultipleNegativesRankingLoss(model=model, mini_batch_size=16)
else:
    loss = losses.MultipleNegativesRankingLoss(model=model)

## 6. Параметры обучения

Гиперпараметры — стартовый набор; на полном датасете подкручивай LR и эпохи под себя.

- `learning_rate=2e-5` — типичен для contrastive fine-tuning encoder-моделей.
- `num_train_epochs=3–5` — больше обычно ведёт к переобучению на узкий домен.
- `warmup_ratio=0.1` — стабилизирует начало.
- `batch_sampler=NO_DUPLICATES` — гарантирует, что в одном батче не встретятся два одинаковых позитива (иначе in-batch negatives ломаются).

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,         # уменьшить до 8–16 для bge-m3 / Qwen3
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,                              # на T4 OK; на A100 лучше bf16=True
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_legal-eval_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",                      # "wandb" если хочешь логировать
    gradient_checkpointing=False,           # включить для bge-m3 / Qwen3 на T4
    seed=SEED,
)

## 7. Trainer и запуск обучения

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss,
    evaluator=ir_evaluator,
)

trainer.train()

## 8. Метрики ПОСЛЕ дообучения и сравнение

In [ ]:
tuned_metrics = ir_evaluator(model)

print("\n=== СРАВНЕНИЕ ===")
print(f"{'metric':55s} {'baseline':>10s} {'tuned':>10s} {'delta':>10s}")
print("-" * 90)
for k in sorted(baseline_metrics.keys()):
    b = baseline_metrics[k]
    t = tuned_metrics.get(k, float("nan"))
    delta = t - b
    print(f"{k:55s} {b:10.4f} {t:10.4f} {delta:+10.4f}")

## 9. Сохранение модели

Лучший чекпойнт уже сохранён в `OUTPUT_DIR` (благодаря `load_best_model_at_end=True`). Также можно запушить в HuggingFace Hub.

In [ ]:
FINAL_DIR = f"{OUTPUT_DIR}/final"
model.save(FINAL_DIR)
print("Сохранено в:", FINAL_DIR)

# Опционально — пуш на Hub
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/multilingual-e5-base-ru-legal")

## 10. Использование в RAG-пайплайне

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("./finetuned-multilingual-e5-base-legal-ru/final")
model.max_seq_length = 512

# Не забудь те же префиксы, на которых обучался!
query_emb = model.encode(["query: какой штраф за обгон через сплошную?"], normalize_embeddings=True)
doc_embs  = model.encode([f"passage: {chunk}" for chunk in chunks], normalize_embeddings=True)
```

## Что улучшить дальше (для отчёта)

1. **Hard negative mining.** После первой эпохи обучения прогоняй тренинговые запросы через текущую модель, выбирай top-k самых похожих документов, не являющихся положительными, и добавляй их в датасет как hard negatives. Даёт +2–5 пунктов NDCG@10 на правовом домене.
2. **Matryoshka-loss.** `MatryoshkaLoss` поверх MNRL обучает эмбеддинги так, чтобы их можно было обрезать до 256/128 размерности с минимальной потерей — экономит память векторной БД.
3. **Comparison across base models.** Логируй метрики во все эксперименты в одну таблицу — это и есть сравнение эмбеддеров из плана проекта.
4. **Реранкер.** Дообученный эмбеддер top-50 → cross-encoder реранкер top-10. Реранкер обучается на тех же `(query, positive, hard_negative)` через `CrossEncoder` + `BinaryCrossEntropyLoss`. Метрики ранжирования замеряются на обоих этапах (как и указано в плане проекта).
